# Обучение CLIP на датасете AI2D
Этот ноутбук демонстрирует, как подготовить данные и обучить модель **CLIP** на датасете AI2D.
CLIP (Contrastive Language–Image Pre‑Training) — модель, которая обучается сопоставлять изображения и подписи,
чтобы разместить их в едином пространстве признаков. Здесь мы используем его для поиска технических схем по текстовому описанию и наоборот.

## Импорт библиотек и определение датасета
Начнём с импорта необходимых библиотек и определения класса `Ai2dClipDataset`, который загружает пары (изображение, текст) из структуры папок AI2D.

In [ ]:

import math
import json
from pathlib import Path
from typing import Tuple, List, Dict
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import resnet50, ResNet50_Weights
from transformers import (
    CLIPModel,
    CLIPProcessor,
    get_cosine_schedule_with_warmup,
)
from PIL import Image
import torch.optim as optim
from tqdm import tqdm
from typing import List, Tuple, Dict, Optional



In [2]:
class Ai2dClipDataset(Dataset):
    """
    Возвращает (image, text, group_id), где group_id = imageName.
    """

    def __init__(self, root_dir: str, text_mode: str = "q+correct") -> None:
        self.root = Path(root_dir)
        self.images_dir = self.root / "images"
        self.questions_dir = self.root / "questions"
        self.text_mode = text_mode

        self.samples: List[Tuple[str, str, str]] = []  

        json_files = sorted(self.questions_dir.glob("*.json"))
        if not json_files:
            raise RuntimeError(f"No question JSONs found in {self.questions_dir}")

        for q_path in json_files:
            with q_path.open("r", encoding="utf-8") as f:
                data = json.load(f)

            image_name = data["imageName"]
            img_path = self.images_dir / image_name
            if not img_path.exists():
                continue

            for q_text, q_data in data["questions"].items():
                answers = q_data.get("answerTexts", [])
                correct_idx = int(q_data.get("correctAnswer", 0))
                if not answers:
                    continue
                if correct_idx < 0 or correct_idx >= len(answers):
                    correct_idx = 0

                if self.text_mode == "q_only":
                    caption = q_text
                elif self.text_mode == "q+options+correct":
                    options_str = "; ".join(f"{chr(ord('A') + i)}. {a}" for i, a in enumerate(answers))
                    caption = f"Question: {q_text} Options: {options_str}. Correct answer: {answers[correct_idx]}."
                else:
                    caption = f"Question: {q_text} Correct answer: {answers[correct_idx]}."

                self.samples.append((str(img_path), caption, image_name))

        print(f"[AI2D] Loaded {len(self.samples)} (image, text) pairs")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img_path, text, group_id = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        return image, text, group_id


## Вспомогательные функции и метрики
Ниже определены функции для фиксирования генераторов случайных чисел, подготовки батчей для CLIP,
контрастивной функции потерь и вычисления метрик Recall@k. Эти функции понадобятся при обучении.

In [3]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def contrastive_loss(logits_per_image: torch.Tensor, logits_per_text: torch.Tensor) -> torch.Tensor:
    bs = logits_per_image.size(0)
    targets = torch.arange(bs, device=logits_per_image.device)
    loss_i2t = F.cross_entropy(logits_per_image, targets)
    loss_t2i = F.cross_entropy(logits_per_text, targets)
    return 0.5 * (loss_i2t + loss_t2i)


def recall_at_k_group(sim: torch.Tensor, group_ids: List[str], ks=(1, 5, 10)) -> Dict[int, float]:
    """
    group_ids: длины N, одинаковые id для всех пар одного изображения (например имя файла).
    Позитивы для строки i: все j, где group_ids[j] == group_ids[i].
    """
    n = sim.size(0)
    ranks = sim.argsort(dim=1, descending=True)  
    g = np.asarray(group_ids)

    recalls = {}
    topk_all = ranks.detach().cpu().numpy()
    for k in ks:
        k = min(k, n)
        topk = topk_all[:, :k]               
        hit = (g[topk] == g[:, None]).any(1) 
        recalls[k] = float(hit.mean())
    return recalls



def clip_collate_fn(batch, processor: CLIPProcessor):
    """
    Поддерживает два формата:
      - (image, text)
      - (image, text, group_id)
    """
    if len(batch[0]) == 3:
        images, texts, group_ids = zip(*batch)
    else:
        images, texts = zip(*batch)
        group_ids = None

    text_inputs = processor.tokenizer(list(texts), return_tensors="pt", padding=True, truncation=True)
    image_inputs = processor.image_processor(list(images), return_tensors="pt")

    out = {**text_inputs, **image_inputs}
    if group_ids is not None:
        out["group_ids"] = list(group_ids)  
    return out

## Функция обучения
Функция `train_ai2d_clip` объединяет все вышеперечисленные компоненты: загружает датасет, делит его на обучающую
и валидационную части, инициализирует модель CLIP и оптимизатор, и запускает цикл обучения.

In [ ]:

def train_one_epoch(
    model: CLIPModel,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler,
    device: torch.device,
    scaler: torch.amp.GradScaler,
    use_amp: bool,
    epoch_idx: int,
    epochs: int,
) -> float:
    model.train()
    running = 0.0

    for batch in tqdm(loader, desc=f"Train {epoch_idx+1}/{epochs}", leave=False):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items() if k != "group_ids"}

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(**batch)
            loss = contrastive_loss(outputs.logits_per_image, outputs.logits_per_text)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running += float(loss.item())

    return running / max(1, len(loader))


@torch.no_grad()
def validate_retrieval(
    model: CLIPModel,
    loader: DataLoader,
    device: torch.device,
    use_amp: bool,
    ks=(1, 5, 10),
    max_items: Optional[int] = None,
):
    """
    Корректно для AI2D: Recall@K считается по group_id (одна картинка -> много текстов).
    Возвращает:
      r_i2t, r_t2i, r_mean  (dict[int,float])
    """
    model.eval()

    image_feats_list = []
    text_feats_list = []
    group_ids_all: List[str] = []
    total = 0

    for batch in tqdm(loader, desc="Val", leave=False):
        group_ids = batch.get("group_ids", None)

        batch_t = {k: v.to(device, non_blocking=True) for k, v in batch.items() if k != "group_ids"}

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            img_feats = model.get_image_features(pixel_values=batch_t["pixel_values"])
            txt_feats = model.get_text_features(
                input_ids=batch_t["input_ids"],
                attention_mask=batch_t.get("attention_mask", None),
            )

        img_feats = F.normalize(img_feats, dim=-1)
        txt_feats = F.normalize(txt_feats, dim=-1)

        image_feats_list.append(img_feats.detach().cpu())
        text_feats_list.append(txt_feats.detach().cpu())

        if group_ids is not None:
            group_ids_all.extend(group_ids)
        else:
            # fallback: строгий 1-1 (обычно для AI2D не лучший вариант)
            group_ids_all.extend([str(i) for i in range(img_feats.size(0))])

        total += img_feats.size(0)
        if max_items is not None and total >= max_items:
            break

    image_feats = torch.cat(image_feats_list, dim=0)
    text_feats = torch.cat(text_feats_list, dim=0)

    if max_items is not None:
        image_feats = image_feats[:max_items]
        text_feats = text_feats[:max_items]
        group_ids_all = group_ids_all[:max_items]

    image_feats = image_feats.to(device)
    text_feats = text_feats.to(device)

    sim_i2t = image_feats @ text_feats.t()  
    sim_t2i = sim_i2t.t()

    r_i2t = recall_at_k_group(sim_i2t, group_ids_all, ks=ks)
    r_t2i = recall_at_k_group(sim_t2i, group_ids_all, ks=ks)
    r_mean = {k: 0.5 * (r_i2t[k] + r_t2i[k]) for k in r_i2t.keys()}

    return r_i2t, r_t2i, r_mean


In [ ]:
set_seed(42)

root_dir = "ai2d"
epochs = 50
batch_size = 8
lr = 1e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = (device.type == "cuda")

dataset = Ai2dClipDataset(root_dir=root_dir, text_mode="q+correct")
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    collate_fn=lambda b: clip_collate_fn(b, processor),
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    collate_fn=lambda b: clip_collate_fn(b, processor),
)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
total_steps = epochs * max(1, len(train_loader))
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

scaler = torch.amp.GradScaler(enabled=use_amp)

for epoch in range(epochs):
    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        scaler=scaler,
        use_amp=use_amp,
        epoch_idx=epoch,
        epochs=epochs,
    )

    r_i2t, r_t2i, r_mean = validate_retrieval(
        model=model,
        loader=val_loader,
        device=device,
        use_amp=use_amp,
        ks=(1, 5, 10),
        max_items=None,   
    )

    print(
        f"Epoch {epoch+1}/{epochs} | loss={train_loss:.4f}\n"
        f"  Image->Text: R@1={r_i2t[1]:.4f} R@5={r_i2t[5]:.4f} R@10={r_i2t[10]:.4f}\n"
        f"  Text->Image: R@1={r_t2i[1]:.4f} R@5={r_t2i[5]:.4f} R@10={r_t2i[10]:.4f}\n"
        f"  Mean:       R@1={r_mean[1]:.4f} R@5={r_mean[5]:.4f} R@10={r_mean[10]:.4f}"
    )


[AI2D] Loaded 15501 (image, text) pairs


Train 1/50:   0%|          | 0/1550 [00:00<?, ?it/s]

C:\Users\Jet\anaconda3\Lib\site-packages\torch\optim\lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


Val:   0%|          | 0/388 [00:00<?, ?it/s]

Epoch 1/50 | loss=0.7168
  Image->Text: R@1=0.1412 R@5=0.3070 R@10=0.4199
  Text->Image: R@1=0.0990 R@5=0.1983 R@10=0.2796
  Mean:       R@1=0.1201 R@5=0.2527 R@10=0.3497


...

Epoch 42/50 | loss=0.0121
  Image->Text: R@1=0.1851 R@5=0.4331 R@10=0.5550
  Text->Image: R@1=0.1448 R@5=0.2806 R@10=0.3937
  Mean:       R@1=0.1649 R@5=0.3568 R@10=0.4744


Train 43/50:   0%|          | 0/1550 [00:00<?, ?it/s]

Val:   0%|          | 0/388 [00:00<?, ?it/s]

Epoch 43/50 | loss=0.0115
  Image->Text: R@1=0.1880 R@5=0.4344 R@10=0.5521
  Text->Image: R@1=0.1445 R@5=0.2828 R@10=0.3908
  Mean:       R@1=0.1662 R@5=0.3586 R@10=0.4715


Train 44/50:   0%|          | 0/1550 [00:00<?, ?it/s]